In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
import json

# PROJECT PATHS

PROJECT_ROOT = Path("/content/drive/MyDrive/intentmap-nids/intentmap-nids")
GNS3_DIR = PROJECT_ROOT / "data"
INPUT_FILE = GNS3_DIR / "all_traffic_features_raw.tsv"

PREPROCESSOR_FILE = (PROJECT_ROOT / "models" / "preprocessor.joblib")
FEATURE_ORDER_FILE = (PROJECT_ROOT / "config" / "feature_order.json")
OUTPUT_21 = (GNS3_DIR / "gns3_all_traffic_21_features.csv")
OUTPUT_41 = (GNS3_DIR / "gns3_all_traffic_41_features.csv")
OUTPUT_METADATA = (GNS3_DIR / "gns3_all_traffic_metadata.csv")


print("Input exists:", INPUT_FILE.exists())
print("Preprocessor exists:", PREPROCESSOR_FILE.exists())
print("Feature order exists:", FEATURE_ORDER_FILE.exists())


Input exists: True
Preprocessor exists: True
Feature order exists: True


In [4]:
#  LOAD FINAL NORMAL + ATTACK DATA

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"File not found:\n{INPUT_FILE}"
    )

df = pd.read_csv(
    INPUT_FILE,
    sep="\t",
    dtype=str,
    keep_default_na=False
)

print("RAW DATA")

print("Shape:", df.shape)
print("\nTraffic types:")
print(df["traffic_type"].value_counts())

print("\nColumns:")
print(df.columns.tolist())

RAW DATA
Shape: (1416, 14)

Traffic types:
traffic_type
nmap_scan       1002
http_burst       200
dns_brust        199
normal            12
bulk_tranfer       3
Name: count, dtype: int64

Columns:
['traffic_type', 'ts', 'id.orig_h', 'id.orig_p', 'id.resp_h', 'id.resp_p', 'proto', 'service', 'duration', 'orig_ip_bytes', 'resp_ip-bytes', 'orig_pkts', 'resp_pkts', 'conn_state']


In [29]:
display(df.head(3))

,traffic_type,ts,id.orig_h,id.orig_p,id.resp_h,id.resp_p,proto,service,duration,orig_ip_bytes,resp_ip-bytes,orig_pkts,resp_pkts,conn_state
0,normal,1786931724.997899,192.168.10.10,54104,192.168.10.20,80,tcp,http,0.004304,345,508,5,5,SF
1,normal,1786931729.272748,192.168.10.10,54108,192.168.10.20,80,tcp,http,0.002353,355,508,5,5,SF
2,normal,1786931732.830067,192.168.10.10,53628,192.168.10.20,80,tcp,http,0.002170,345,508,5,5,SF


In [6]:
print(df.columns.tolist())
print("resp_ip_bytes exists:", "resp_ip_bytes" in df.columns)

['traffic_type', 'ts', 'id.orig_h', 'id.orig_p', 'id.resp_h', 'id.resp_p', 'proto', 'service', 'duration', 'orig_ip_bytes', 'resp_ip_bytes', 'orig_pkts', 'resp_pkts', 'conn_state']
resp_ip_bytes exists: True


In [8]:
# verifying columns we need from the Zeek data

required_columns = [
    "traffic_type",
    "ts",
    "id.orig_h",
    "id.orig_p",
    "id.resp_h",
    "id.resp_p",
    "proto",
    "service",
    "duration",
    "orig_ip_bytes",
    "resp_ip_bytes",
    "orig_pkts",
    "resp_pkts",
    "conn_state"
]


# Check if anything is missing

missing = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing:
    raise ValueError(f"Missing columns: {missing}")

print("All required columns are available.")


# Convert numerical fields into numbers

numeric_columns = [
    "ts",
    "id.orig_p",
    "id.resp_p",
    "duration",
    "orig_ip_bytes",
    "resp_ip_bytes",
    "orig_pkts",
    "resp_pkts"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    ).fillna(0)


# Clean text values

df["proto"] = (
    df["proto"]
    .str.strip()
    .str.lower()
)

df["service"] = df["service"].str.strip()
df["conn_state"] = df["conn_state"].str.strip()


# Replace empty values with useful labels

df.loc[
    df["service"].isin(["", "(empty)"]),
    "service"
] = "-"

df.loc[
    df["proto"] == "",
    "proto"
] = "other"

df.loc[
    df["conn_state"] == "",
    "conn_state"
] = "OTH"


# Keep traffic in time order inside each scenario

df = (
    df.sort_values(
        ["traffic_type", "ts"]
    )
    .reset_index(drop=True)
)

print("Data cleaning complete.")

All required columns are available.
Data cleaning complete.


In [9]:
# Start a new dataframe for the 21 model features

X21 = pd.DataFrame(index=df.index)


# Direct features from Zeek

X21["dur"] = df["duration"]
X21["spkts"] = df["orig_pkts"]
X21["dpkts"] = df["resp_pkts"]

# Use IP bytes because this matches our feature dictionary
X21["sbytes"] = df["orig_ip_bytes"]
X21["dbytes"] = df["resp_ip_bytes"]


# Total packet rate

X21["rate"] = np.where(
    df["duration"] > 0,
    (df["orig_pkts"] + df["resp_pkts"])
    / df["duration"],
    0
)


# Source traffic load in bits per second

X21["sload"] = np.where(
    df["duration"] > 0,
    (df["orig_ip_bytes"] * 8)
    / df["duration"],
    0
)


# Destination traffic load

X21["dload"] = np.where(
    df["duration"] > 0,
    (df["resp_ip_bytes"] * 8)
    / df["duration"],
    0
)


# Average source packet size

X21["smean"] = np.where(
    df["orig_pkts"] > 0,
    df["orig_ip_bytes"]
    / df["orig_pkts"],
    0
)


# Average destination packet size

X21["dmean"] = np.where(
    df["resp_pkts"] > 0,
    df["resp_ip_bytes"]
    / df["resp_pkts"],
    0
)


print("Basic network features created.")

display(X21.head())

Basic network features created.


,dur,spkts,dpkts,sbytes,dbytes,rate,sload,dload,smean,dmean
0,80.667625,19,17,4573,4016,0.446276,4.535153e+02,3.982763e+02,240.684211,236.235294
1,112.978533,15,12,4005,3652,0.238983,2.835937e+02,2.585978e+02,267.000000,304.333333
2,13.305999,55655,26023,107883229,1419792,6138.434251,6.486291e+07,8.536252e+05,1938.428335,54.559121
3,0.000432,1,1,76,80,4629.629630,1.407407e+06,1.481481e+06,76.000000,80.000000
4,0.000469,1,1,76,80,4264.392324,1.296375e+06,1.364606e+06,76.000000,80.000000


In [10]:
# These features describe recent connection behaviour

count_features = [
    "ct_srv_src",
    "ct_srv_dst",
    "ct_dst_ltm",
    "ct_src_ltm",
    "ct_src_dport_ltm",
    "ct_dst_sport_ltm",
    "ct_dst_src_ltm"
]


# Create empty columns first

for feature in count_features:
    X21[feature] = 0


def calculate_recent_connections(group):
    """
    Calculate connection-count features using the
    current connection and up to 99 previous connections.
    """

    results = pd.DataFrame(
        0,
        index=group.index,
        columns=count_features
    )

    rows = group.reset_index()

    for i in range(len(rows)):

        # Look at the current connection and previous 99
        start = max(0, i - 99)

        window = rows.iloc[start:i + 1]
        current = rows.iloc[i]

        row_index = current["index"]


        # Same service and same source IP
        results.loc[row_index, "ct_srv_src"] = (
            (
                (window["service"] == current["service"]) &
                (window["id.orig_h"] == current["id.orig_h"])
            ).sum()
        )


        # Same service and same destination IP
        results.loc[row_index, "ct_srv_dst"] = (
            (
                (window["service"] == current["service"]) &
                (window["id.resp_h"] == current["id.resp_h"])
            ).sum()
        )


        # Same destination IP
        results.loc[row_index, "ct_dst_ltm"] = (
            window["id.resp_h"]
            .eq(current["id.resp_h"])
            .sum()
        )


        # Same source IP
        results.loc[row_index, "ct_src_ltm"] = (
            window["id.orig_h"]
            .eq(current["id.orig_h"])
            .sum()
        )


        # Same source IP and destination port
        results.loc[row_index, "ct_src_dport_ltm"] = (
            (
                window["id.orig_h"].eq(current["id.orig_h"]) &
                window["id.resp_p"].eq(current["id.resp_p"])
            ).sum()
        )


        # Same destination IP and source port
        results.loc[row_index, "ct_dst_sport_ltm"] = (
            (
                window["id.resp_h"].eq(current["id.resp_h"]) &
                window["id.orig_p"].eq(current["id.orig_p"])
            ).sum()
        )


        # Same source and destination IP pair
        results.loc[row_index, "ct_dst_src_ltm"] = (
            (
                window["id.orig_h"].eq(current["id.orig_h"]) &
                window["id.resp_h"].eq(current["id.resp_h"])
            ).sum()
        )

    return results

In [11]:
# Calculate the recent-connection features separately
# for normal traffic and each attack capture

for traffic_type in df["traffic_type"].unique():

    group_index = df.index[
        df["traffic_type"] == traffic_type
    ]

    group = df.loc[group_index]

    recent_counts = calculate_recent_connections(group)

    for feature in count_features:
        X21.loc[
            recent_counts.index,
            feature
        ] = recent_counts[feature]


print("Recent connection features created.")

display(
    X21[count_features].head(10)
)

Recent connection features created.


,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm
0,1,1,1,1,1,1,1
1,2,2,2,2,2,1,2
2,3,3,3,3,3,1,3
3,1,1,1,1,1,1,1
4,2,2,2,2,2,1,2
5,3,3,3,3,3,1,3
6,4,4,4,4,4,1,4
7,5,5,5,5,5,1,5
8,6,6,6,6,6,1,6
9,7,7,7,7,7,1,7


In [12]:
# Check whether source and destination
# IP address and port are the same

X21["is_sm_ips_ports"] = (
    (df["id.orig_h"] == df["id.resp_h"]) &
    (df["id.orig_p"] == df["id.resp_p"])
).astype(int)


# Keep the protocols used by our fitted preprocessor.
# Anything else becomes "other".

def map_protocol(protocol):

    if protocol in ["tcp", "udp", "arp"]:
        return protocol

    return "other"


X21["proto"] = df["proto"].apply(map_protocol)


# Service can be copied directly

X21["service"] = df["service"]


# Zeek and UNSW-NB15 use different state names.
# This mapping is therefore an approximation.

def map_connection_state(row):

    zeek_state = row["conn_state"]
    protocol = row["proto"]

    if protocol == "icmp":
        return "ECO"

    if zeek_state == "SF":
        return "FIN"

    if zeek_state in ["S1", "S2", "S3"]:
        return "CON"

    if zeek_state in ["S0", "SH", "SHR"]:
        return "REQ"

    if zeek_state in [
        "REJ",
        "RSTO",
        "RSTR",
        "RSTOS0",
        "RSTRH"
    ]:
        return "RST"

    return "no"


X21["state"] = df.apply(
    map_connection_state,
    axis=1
)


print("Protocol, service and state features added.")

Protocol, service and state features added.


In [13]:
# Exact 21 portable features used by our project

FEATURES_21 = [
    "dur",
    "spkts",
    "dpkts",
    "sbytes",
    "dbytes",
    "rate",
    "sload",
    "dload",
    "smean",
    "dmean",
    "ct_srv_src",
    "ct_srv_dst",
    "ct_dst_ltm",
    "ct_src_ltm",
    "ct_src_dport_ltm",
    "ct_dst_sport_ltm",
    "ct_dst_src_ltm",
    "is_sm_ips_ports",
    "proto",
    "service",
    "state"
]


X21 = X21[FEATURES_21]


# Replace invalid numerical values if any appear

numeric_features = [
    feature
    for feature in FEATURES_21
    if feature not in ["proto", "service", "state"]
]

X21[numeric_features] = (
    X21[numeric_features]
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)


print("21-feature dataset shape:", X21.shape)
print("Missing values:", X21.isna().sum().sum())

display(X21.head())

21-feature dataset shape: (1416, 21)
Missing values: 0


,dur,spkts,dpkts,sbytes,dbytes,rate,sload,dload,smean,dmean,...,ct_srv_dst,ct_dst_ltm,ct_src_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,is_sm_ips_ports,proto,service,state
0,80.667625,19,17,4573,4016,0.446276,4.535153e+02,3.982763e+02,240.684211,236.235294,...,1,1,1,1,1,1,0,tcp,ssh,FIN
1,112.978533,15,12,4005,3652,0.238983,2.835937e+02,2.585978e+02,267.000000,304.333333,...,2,2,2,2,1,2,0,tcp,ssh,FIN
2,13.305999,55655,26023,107883229,1419792,6138.434251,6.486291e+07,8.536252e+05,1938.428335,54.559121,...,3,3,3,3,1,3,0,tcp,ssh,FIN
3,0.000432,1,1,76,80,4629.629630,1.407407e+06,1.481481e+06,76.000000,80.000000,...,1,1,1,1,1,1,0,udp,dns,FIN
4,0.000469,1,1,76,80,4264.392324,1.296375e+06,1.364606e+06,76.000000,80.000000,...,2,2,2,2,1,2,0,udp,dns,FIN


In [14]:
# which original connection each prediction belongs to

metadata = pd.DataFrame({
    "traffic_type": df["traffic_type"],
    "ts": df["ts"],
    "src_ip": df["id.orig_h"],
    "src_port": df["id.orig_p"],
    "dst_ip": df["id.resp_h"],
    "dst_port": df["id.resp_p"]
})


# 0 = normal capture
# 1 = controlled attack capture

metadata["label"] = (
    metadata["traffic_type"] != "normal"
).astype(int)


# Mark connections that actually started from Kali

metadata["kali_source"] = (
    metadata["src_ip"] == "192.168.10.30"
).astype(int)


print("Traffic labels:")
print(metadata["traffic_type"].value_counts())

print("\nBinary labels:")
print(metadata["label"].value_counts())

display(metadata.head())

Traffic labels:
traffic_type
nmap_scan       1002
http_burst       200
dns_brust        199
normal            12
bulk_tranfer       3
Name: count, dtype: int64

Binary labels:
label
1    1404
0      12
Name: count, dtype: int64


,traffic_type,ts,src_ip,src_port,dst_ip,dst_port,label,kali_source
0,bulk_tranfer,1.787879e+09,192.168.10.30,58670,192.168.10.20,22,1,1
1,bulk_tranfer,1.787879e+09,192.168.10.30,34528,192.168.10.20,22,1,1
2,bulk_tranfer,1.787879e+09,192.168.10.30,41100,192.168.10.20,22,1,1
3,dns_brust,1.787883e+09,192.168.10.30,40311,192.168.10.20,53,1,1
4,dns_brust,1.787883e+09,192.168.10.30,45869,192.168.10.20,53,1,1


In [15]:
# Combine metadata with the 21 model features

final_21 = pd.concat(
    [
        metadata.reset_index(drop=True),
        X21.reset_index(drop=True)
    ],
    axis=1)

OUTPUT_21 = (
    GNS3_DIR /
    "gns3_all_traffic_21_features.csv")

OUTPUT_METADATA = (
    GNS3_DIR /
    "gns3_all_traffic_metadata.csv")

final_21.to_csv(
    OUTPUT_21,
    index=False)

metadata.to_csv(
    OUTPUT_METADATA,
    index=False)


print("21-feature file saved to:")
print(OUTPUT_21)

print("\nMetadata saved to:")
print(OUTPUT_METADATA)

21-feature file saved to:
/content/drive/MyDrive/intentmap-nids/intentmap-nids/data/gns3_all_traffic_21_features.csv

Metadata saved to:
/content/drive/MyDrive/intentmap-nids/intentmap-nids/data/gns3_all_traffic_metadata.csv


In [16]:
# Load the preprocessing pipeline that was
# already fitted using the UNSW-NB15 training data

if not PREPROCESSOR_FILE.exists():

    raise FileNotFoundError(
        f"Preprocessor not found: {PREPROCESSOR_FILE}"
    )


preprocessor = joblib.load(
    PREPROCESSOR_FILE
)


print("Preprocessor loaded successfully.")
print(PREPROCESSOR_FILE)

Preprocessor loaded successfully.
/content/drive/MyDrive/intentmap-nids/intentmap-nids/models/preprocessor.joblib


In [17]:
# Apply the existing preprocessing pipeline

X41_raw = preprocessor.transform(X21)


# Some sklearn transformers return a sparse matrix,
# so convert it to a normal NumPy array if needed.

if hasattr(X41_raw, "toarray"):
    X41_array = X41_raw.toarray()
else:
    X41_array = np.asarray(X41_raw)


print("Transformed shape:", X41_array.shape)

Transformed shape: (1416, 41)


In [18]:
FEATURES_41 = [
    "dur",
    "spkts",
    "dpkts",
    "sbytes",
    "dbytes",
    "rate",
    "sload",
    "dload",
    "smean",
    "dmean",
    "ct_srv_src",
    "ct_srv_dst",
    "ct_dst_ltm",
    "ct_src_ltm",
    "ct_src_dport_ltm",
    "ct_dst_sport_ltm",
    "ct_dst_src_ltm",
    "is_sm_ips_ports",

    "proto_arp",
    "proto_other",
    "proto_tcp",
    "proto_udp",

    "service_-",
    "service_dns",
    "service_ftp",
    "service_ftp-data",
    "service_http",
    "service_pop3",
    "service_radius",
    "service_smtp",
    "service_snmp",
    "service_ssh",

    "state_CON",
    "state_ECO",
    "state_FIN",
    "state_INT",
    "state_PAR",
    "state_REQ",
    "state_RST",
    "state_URN",
    "state_no"
]


if X41_array.shape[1] != len(FEATURES_41):

    raise ValueError(
        f"Expected 41 features but got "
        f"{X41_array.shape[1]}"
    )


X41 = pd.DataFrame(
    X41_array,
    columns=FEATURES_41
)


print("Final model feature shape:", X41.shape)

display(X41.head())

Final model feature shape: (1416, 41)


,dur,spkts,dpkts,sbytes,dbytes,rate,sload,dload,smean,dmean,...,service_ssh,state_CON,state_ECO,state_FIN,state_INT,state_PAR,state_REQ,state_RST,state_URN,state_no
0,8.554070,0.126045,0.084731,0.569438,0.245787,-1.615072,-1.804966,-1.360752,1.518415,0.488058,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1,9.208567,0.028277,-0.066504,0.500037,0.219637,-1.650930,-1.926114,-1.441665,1.651011,0.615129,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2,5.133906,3.601016,3.466333,5.839259,1.861180,0.321181,1.266265,0.080320,4.190105,-0.242891,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3,-0.089167,-0.882807,-0.936397,-1.568310,-0.828944,0.255807,0.275038,0.183934,0.050983,-0.053054,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,-0.089094,-0.882807,-0.936397,-1.568310,-0.828944,0.236763,0.253773,0.168490,0.050983,-0.053054,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [19]:
OUTPUT_41 = (
    GNS3_DIR /
    "gns3_all_traffic_41_features.csv"
)


X41.to_csv(
    OUTPUT_41,
    index=False
)


print("41-feature model dataset saved to:")
print(OUTPUT_41)

41-feature model dataset saved to:
/content/drive/MyDrive/intentmap-nids/intentmap-nids/data/gns3_all_traffic_41_features.csv


In [25]:
print("=" * 55)
print("GNS3 DATA PREPARATION COMPLETE")
print("=" * 55)

print("\nRaw Zeek records:")
print(df.shape)

print("\n21 portable features:")
print(X21.shape)

print("\n41 model features:")
print(X41.shape)

print("\nTraffic types:")
print(metadata["traffic_type"].value_counts())

print("\nMissing values in 21 features:")
print(X21.isna().sum().sum())

print("\nMissing values in 41 features:")
print(X41.isna().sum().sum())

print("\nSaved files:")
print("1.", OUTPUT_21)
print("2.", OUTPUT_41)
print("3.", OUTPUT_METADATA)

GNS3 DATA PREPARATION COMPLETE

Raw Zeek records:
(1416, 14)

21 portable features:
(1416, 21)

41 model features:
(1416, 41)

Traffic types:
traffic_type
nmap_scan        1002
http_burst        200
dns_brust         199
normal             12
bulk_transfer       3
Name: count, dtype: int64

Missing values in 21 features:
0

Missing values in 41 features:
0

Saved files:
1. /content/drive/MyDrive/intentmap-nids/intentmap-nids/data/gns3_all_traffic_21_features.csv
2. /content/drive/MyDrive/intentmap-nids/intentmap-nids/data/gns3_all_traffic_41_features.csv
3. /content/drive/MyDrive/intentmap-nids/intentmap-nids/data/gns3_all_traffic_metadata.csv
